<a href="https://colab.research.google.com/github/rhyan10/X-MACE/blob/X-MACE_socs/tutorial-summer-school.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# X-MACE Tutorial: From Dataset to Trained Model

**X-MACE** models excited-state potential energy surfaces — including conical
intersections, non-adiabatic couplings (NACs) and spin-orbit couplings (SOCs). It extends
[MACE](https://github.com/ACEsuit/mace) with **Deep Sets**, giving a smooth representation
of inherently non-smooth multi-state surfaces.

**Part 1 — The data** (sections 1-6)
Read an extended XYZ file, understand `atoms.info`, verify shapes, and look at the
conical-intersection region.

**Part 2 — Training** (sections 7-12)
Install X-MACE, understand the two model variants, build a correct training command,
run a short job, and read the results.

Everything is **derived from the dataset itself** — atom count, number of states, which
properties exist, and which key names they use. Point `DATA_FILE` at a different file and
both the analysis and the training command adapt automatically.

> Barrett et al., [arXiv:2502.12870](https://arxiv.org/abs/2502.12870) (2025) ·
> [github.com/rhyan10/X-MACE](https://github.com/rhyan10/X-MACE)

---

# Part 1 — The Dataset

## 1. Setup

Colab starts from a clean machine every time, so install ASE and check the hardware here.

In [ ]:
!pip install -q ase

import ase, ase.io
import numpy as np

print("ASE version:", ase.__version__)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("PyTorch    :", torch.__version__)
except ImportError:
    DEVICE = "cpu"

print("Device     :", DEVICE)
if DEVICE == "cpu":
    print("\nNo GPU available. Everything here still works — training is just slower.")
    print("To request one: Runtime > Change runtime type > T4 GPU.")

### 1.1 Get the dataset

Tries in order: a file already present, a download from `DATA_URL`, then manual upload.

In [ ]:
import os

DATA_FILE = "ch2nh2.xyz"
DATA_URL  = "https://raw.githubusercontent.com/rhyan10/X-MACE/X-MACE_socs/tutorials/ch2nh2.xyz"

if os.path.exists(DATA_FILE):
    print(f"Found {DATA_FILE} already.")
else:
    got = False
    if DATA_URL:
        rc = os.system(f"wget -q -O {DATA_FILE} {DATA_URL}")
        got = rc == 0 and os.path.getsize(DATA_FILE) > 0
        print("Downloaded." if got else "Download failed — falling back to upload.")
    if not got:
        if os.path.exists(DATA_FILE):
            os.remove(DATA_FILE)
        from google.colab import files
        files.upload()

assert os.path.exists(DATA_FILE), f"{DATA_FILE} is missing — cannot continue."
print("Size on disk:", round(os.path.getsize(DATA_FILE) / 1e6, 2), "MB")

## 2. Reading the File

Pass `":"` to read **all** frames, or `":500"` for a faster subset while exploring.

In [ ]:
db = ase.io.read(DATA_FILE, ":")
atoms = db[0]

print(f"Total frames loaded : {len(db)}")
print(f"Type of each element: {type(atoms)}")

## 3. Basic Atoms Properties

In [ ]:
print("Number of atoms    :", len(atoms))
print("Chemical formula   :", atoms.get_chemical_formula())
print("Chemical symbols   :", list(atoms.symbols))
print("Positions shape    :", atoms.positions.shape)   # (N_atoms, 3)

## 4. What's Actually in `atoms.info`?

`atoms.info` is a plain Python `dict` holding everything from the extXYZ comment line.

**Look before you assume.** Key names are not consistent across X-MACE datasets:
`ch2nh2.xyz` stores NACs as `smooth_nacs`, while `SINGLET_SOC_ALL.xyz` uses
`REF_smooth_nacs`. Some datasets have SOCs, some don't. Getting this wrong is the most
common cause of a training run that starts fine and learns nothing.

In [ ]:
print(f"{'key':22s} {'shape':16s} dtype")
print("-" * 50)
for k, v in atoms.info.items():
    arr = np.asarray(v)
    print(f"{k:22s} {str(arr.shape):16s} {arr.dtype}")

## 5. Working Out the Dataset Dimensions

Read the numbers off the data instead of hardcoding them:

* `N` — atoms per frame
* `n_states` — electronic states, from the trailing axis of the energy array
* `n_pairs` — unique state pairs, `n_states x (n_states - 1) / 2`; this is how many NAC
  vectors exist, and what `--nac_num` must be set to

**These variables drive the training command in Part 2**, so this cell is the single
place the whole notebook gets configured.

In [ ]:
from itertools import combinations

ENERGY_KEY = 'REF_energy'
FORCES_KEY = 'REF_forces'

energy   = np.array(atoms.info[ENERGY_KEY])
N        = len(atoms)
n_states = energy.shape[-1]
n_pairs  = n_states * (n_states - 1) // 2

# NAC key naming varies between datasets — detect by substring
present_nacs = [k for k in atoms.info
                if ('nac' in k.lower() or 'coupling' in k.lower())]
present_nacs.sort(key=lambda k: ('smooth' not in k.lower(), k))   # prefer smoothed
NAC_KEY = present_nacs[0] if present_nacs else None

HAS_FORCES = FORCES_KEY in atoms.info
HAS_SOCS   = 'REF_socs' in atoms.info or 'socs' in atoms.info
SOC_KEY    = 'REF_socs' if 'REF_socs' in atoms.info else ('socs' if 'socs' in atoms.info else None)
N_SOC      = int(np.array(atoms.info[SOC_KEY]).size) if SOC_KEY else 0

print(f"N_atoms   : {N}")
print(f"n_states  : {n_states}      -> --n_energies={n_states}")
print(f"n_pairs   : {n_pairs}      -> --nac_num={n_pairs}")
print(f"forces    : {'yes' if HAS_FORCES else 'no'}")
print(f"NAC key   : {NAC_KEY or 'none found'}"
      + (f'   -> --nacs_key="{NAC_KEY}"' if NAC_KEY else ""))
print(f"SOCs      : {f'yes, n_soc={N_SOC}  -> --soc_num={N_SOC}' if HAS_SOCS else 'no'}")

### Expected shapes

| Key | Shape | Meaning |
|---|---|---|
| `REF_energy` | `(1, n_states)` | one geometry x state energies |
| `REF_forces` | `(N, n_states, 3)` | atoms x states x xyz |
| NAC key | `(N, n_pairs, 3)` | atoms x state pairs x xyz |
| SOC key | `(1, n_soc)` | flat SOC vector, if present |

## 6. Inspecting Each Property

### 6.1 Energies

In [ ]:
print("shape  :", energy.shape)
print("values :", energy)

ev = np.sort(energy.ravel())
print("\nsorted :", ev)
print("S1-S0 gap at this geometry:", ev[1] - ev[0])

### 6.2 Forces

In [ ]:
if HAS_FORCES:
    forces = np.array(atoms.info[FORCES_KEY])
    print("forces shape :", forces.shape, f"  expected ({N}, {n_states}, 3)")
    print("state-first  :", forces.transpose(1, 0, 2).shape)
    print("largest |force| in this frame:", np.abs(forces).max())
else:
    print("No forces in this dataset.")

### 6.3 Nonadiabatic Couplings

NACs are indexed by *state pair*, not by state — with `n_states` states there are
`n_pairs` of them, ordered (0,1), (0,2), ...

In [ ]:
if NAC_KEY:
    nacs = np.array(atoms.info[NAC_KEY])
    print(f"{NAC_KEY} shape :", nacs.shape, f"  expected ({N}, {n_pairs}, 3)")

    if n_states == n_pairs:
        print(f"\nNote: with {n_states} states, n_pairs is also {n_pairs}, so forces and")
        print("NACs have identical shapes. Don't rely on shape alone to tell them apart.")

    print("\nstate pairs (column order):")
    for i, (a, b) in enumerate(combinations(range(n_states), 2)):
        print(f"  column {i}:  S{a} - S{b}   |max| = {np.abs(nacs[:, i, :]).max():.6f}")
else:
    print("No NAC key found in this dataset.")

### 6.4 Spin-Orbit Couplings

Only present in datasets that include triplets (e.g. `SINGLET_SOC_ALL.xyz`).

In [ ]:
if HAS_SOCS:
    socs = np.array(atoms.info[SOC_KEY])
    print(f"{SOC_KEY} shape :", socs.shape)
    print("n_soc :", N_SOC, " -> use for --soc_num")
else:
    print("No SOCs in this dataset — omit --compute_socs when training.")

## 7. Sanity Check Across All Frames

Every frame, not just the first — and only for properties this dataset actually has.

In [ ]:
def check_frame(a, verbose=False):
    problems, n = [], len(a)

    exp = {ENERGY_KEY: (1, n_states)}
    if HAS_FORCES:
        exp[FORCES_KEY] = (n, n_states, 3)
    if NAC_KEY:
        exp[NAC_KEY] = (n, n_pairs, 3)
    if HAS_SOCS:
        exp[SOC_KEY] = np.array(db[0].info[SOC_KEY]).shape

    for key, want in exp.items():
        if key not in a.info:
            problems.append(f"{key}: missing")
            if verbose:
                print(f"  [--] {key:22s} missing")
            continue
        got = np.array(a.info[key]).shape
        if got != want:
            problems.append(f"{key}: expected {want}, got {got}")
        if verbose:
            print(f"  [{'OK ' if got == want else 'BAD'}] {key:22s} "
                  f"expected {str(want):16s} got {got}")
    return problems


print("Frame 0 in detail:")
check_frame(db[0], verbose=True)

print("\nScanning all frames...")
bad = {i: p for i, a in enumerate(db) if (p := check_frame(a))}

if not bad:
    print(f"All {len(db)} frames have consistent shapes.")
else:
    print(f"{len(bad)} of {len(db)} frames have problems. First few:")
    for i, probs in list(bad.items())[:5]:
        print(f"  frame {i}: {'; '.join(probs)}")

## 8. The Conical Intersection Region

Near-zero S1-S0 gaps are where the adiabatic surfaces touch — the non-smooth region that
X-MACE exists to handle. The right-hand panel shows why NACs matter: they diverge exactly
where the gap closes.

In [ ]:
import matplotlib.pyplot as plt

gaps, min_e, max_f = [], [], []
for f in db:
    e = np.sort(np.array(f.info[ENERGY_KEY]).ravel())
    min_e.append(e[0])
    gaps.append(e[1] - e[0] if len(e) > 1 else np.nan)
    if HAS_FORCES:
        max_f.append(np.abs(np.array(f.info[FORCES_KEY])).max())

gaps, min_e = np.array(gaps), np.array(min_e)

fig, axes = plt.subplots(1, 2 if NAC_KEY else 1,
                         figsize=(11 if NAC_KEY else 6, 4), squeeze=False)
axes[0][0].hist(gaps, bins=40)
axes[0][0].set_xlabel("S1 - S0 gap")
axes[0][0].set_ylabel("Number of frames")
axes[0][0].set_title("Energy gap distribution")

if NAC_KEY:
    nac_mag = np.array([np.abs(np.array(f.info[NAC_KEY])).max() for f in db])
    axes[0][1].scatter(gaps, nac_mag, s=12, alpha=0.6)
    axes[0][1].set_xlabel("S1 - S0 gap")
    axes[0][1].set_ylabel("max |NAC|")
    axes[0][1].set_yscale("log")
    axes[0][1].set_title("NAC magnitude vs gap")

plt.tight_layout()
plt.show()

print(f"Frames        : {len(db)}")
print(f"Energy range  : {min_e.min():.4f} to {min_e.max():.4f}")
print(f"Smallest gap  : {gaps.min():.6f}  (frame {gaps.argmin()})")
if HAS_FORCES:
    max_f = np.array(max_f)
    print(f"Largest |force|: {max_f.max():.4f}  (frame {max_f.argmax()})")

---

# Part 2 — Training X-MACE

## 9. Installing X-MACE

Cloning plus `pip install .` pulls in `torch`, `e3nn==0.5.1`, `torch-ema`, `matscipy` and
friends. Colab already ships a compatible torch, so this usually takes 2-4 minutes.

Set `INSTALL_XMACE = False` if you only want Part 1, or if you have already run this once
in this session.

> **If it fails:** the usual culprit is a torch/e3nn version clash. `e3nn` is pinned to
> `0.5.1`, so if Colab's torch has moved on, install a matching torch *first*. After any
> install that replaces a preloaded package, use **Runtime > Restart session** before
> continuing.

In [ ]:
INSTALL_XMACE = True   #@param {type:"boolean"}

import os

def xmace_ready():
    """True if the mace package actually imports."""
    import importlib
    try:
        importlib.invalidate_caches()
        importlib.import_module("mace")
        return True
    except ImportError:
        return False

if not INSTALL_XMACE:
    print("Skipping install (INSTALL_XMACE = False).")
elif xmace_ready():
    print("X-MACE is already installed and importable.")
else:
    if not os.path.exists("X-MACE"):
        !git clone -q --depth 1 -b X-MACE_socs https://github.com/rhyan10/X-MACE.git
    !cd X-MACE && pip install .

    if xmace_ready():
        print("\nX-MACE installed and importable.")
    else:
        print("\nInstall finished but 'import mace' still fails.")
        print("This is normal when pip replaced a preloaded package.")
        print("Go to Runtime > Restart session, then re-run this cell.")

## 10. The Two Model Variants

| `--model` | Name | What it does |
|---|---|---|
| `AutoencoderExcitedMACE` | **X-MACE** | Adds a matrix-diagonalisation step via an autoencoder. Better energy accuracy near conical intersections. Energies and forces only. |
| `ExcitedMACE` | **E-MACE** | Standard multi-state readout, no diagonalisation. Faster, and required for NAC and SOC training. |

Since this dataset has NACs, the sections below use `ExcitedMACE`. Switch to
`AutoencoderExcitedMACE` for an energies-and-forces-only comparison.

## 11. Building the Training Command

Rather than copying flags from the README and hoping they match, generate them from what
section 5 detected. This is what prevents the classic failure of training with
`--nac_num=6` on a 3-state dataset, or `--nacs_key="REF_nacs"` on a file whose key is
actually `smooth_nacs`.

Note the X-MACE defaults are `--energy_key=REF_energy`, `--forces_key=REF_forces` and
`--nacs_key=smooth_nacs` — which is exactly what `ch2nh2.xyz` uses.

In [ ]:
#@title Training configuration { run: "auto" }
MODEL        = "ExcitedMACE"   #@param ["ExcitedMACE", "AutoencoderExcitedMACE"]
MAX_EPOCHS   = 5               #@param {type:"slider", min:1, max:200, step:1}
BATCH_SIZE   = 10              #@param {type:"integer"}
HIDDEN       = "32x0e + 32x1o" #@param ["32x0e + 32x1o", "128x0e + 128x1o"]
R_MAX        = 5.0             #@param {type:"number"}
LR           = 0.0001          #@param {type:"number"}
TRAIN_NACS   = True            #@param {type:"boolean"}
NAME         = "tutorial_run"  #@param {type:"string"}

mlp = HIDDEN.split("+")[0].strip()          # MLP_irreps matches the scalar part
use_nacs = TRAIN_NACS and NAC_KEY is not None and MODEL == "ExcitedMACE"

flags = [
    f'--name="{NAME}"',
    f'--train_file="{os.path.abspath(DATA_FILE)}"',
    "--seed=100",
    "--valid_fraction=0.1",
    "--E0s='average'",
    f'--model="{MODEL}"',
    f"--r_max={R_MAX}",
    f"--batch_size={BATCH_SIZE}",
    f"--n_energies={n_states}",          # from the data
    "--correlation=3",
    f"--max_num_epochs={MAX_EPOCHS}",
    "--ema",
    f"--lr={LR}",
    "--ema_decay=0.99",
    '--default_dtype="float32"',
    f"--device={DEVICE}",                # cpu or cuda, detected in section 1
    f'--hidden_irreps="{HIDDEN}"',
    f"--MLP_irreps='{mlp}'",
    "--num_radial_basis=8",
    "--num_interactions=2",
    "--energy_weight=100.0",
    "--forces_weight=100.0" if HAS_FORCES else "--forces_weight=0.0",
    '--error_table="EnergyNacsDipoleMAE"',
]

if use_nacs:
    flags += ["--compute_nacs",
              f"--nac_num={n_pairs}",        # from the data
              f'--nacs_key="{NAC_KEY}"']     # from the data
if HAS_SOCS:
    flags += ["--compute_socs", f"--soc_num={N_SOC}"]

CMD = "python scripts/run_train.py \\\n  " + " \\\n  ".join(flags)
print(CMD)

### What the key flags mean

| Flag | Meaning |
|---|---|
| `--n_energies` | Number of electronic states to model |
| `--r_max` | Cutoff radius in Å — atoms beyond this don't communicate directly |
| `--hidden_irreps` | Representation size and angular order. `128x0e + 128x1o` is large; `32x0e + 32x1o` trains fast |
| `--num_interactions` | Message-passing layers; 2 gives a receptive field of `2 x r_max` |
| `--correlation` | Body order per layer (3 = 4-body) |
| `--energy_weight` / `--forces_weight` / `--nacs_weight` | Relative weight of each term in the loss |
| `--ema` / `--ema_decay` | Exponential moving average of weights — stabilises training |
| `--E0s='average'` | Isolated-atom energies estimated from the dataset average |
| `--compute_nacs` / `--nac_num` / `--nacs_key` | Enable the NAC head, how many vectors, and which dataset key |
| `--foundation_model` | Initialise from a pre-trained ground-state MACE (see section 13) |

**Tip:** to train only one property, keep everything in the dataset and set the other
loss weights to `0.0` — X-MACE expects energies to be present regardless.

## 12. Running It

A few epochs on this dataset is enough to see the loss move — that's the goal here, not a
converged model. On CPU, keep `MAX_EPOCHS` small and `HIDDEN` at `32x0e + 32x1o`.

Output streams below and is also written to `results/`. Checkpoints land in
`checkpoints/`.

In [ ]:
RUN_TRAINING = True   #@param {type:"boolean"}

if RUN_TRAINING and xmace_ready():
    import subprocess, sys
    proc = subprocess.Popen(CMD.replace("\\\n", " "), shell=True, cwd="X-MACE",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print(f"\n--- finished with exit code {proc.returncode} ---")
elif not xmace_ready():
    print("X-MACE is not importable — run section 9, and restart the session if asked.")
else:
    print("Set RUN_TRAINING = True to launch.")

## 13. Transfer Learning from a Foundation Model

X-MACE can start from a pre-trained ground-state MACE (e.g. `medium_off` from MACE-OFF23).
Only the final DeepSets readout is reinitialised — the message-passing kernels, radial
basis and equivariant features are kept. This cuts the amount of excited-state data needed
and generalises better to molecules outside the training set.

Add one flag to the command from section 11:

In [ ]:
transfer_flags = flags + ['--foundation_model="medium_off"']
TRANSFER_CMD = ("python scripts/run_train.py \\\n  "
                + " \\\n  ".join(f.replace(f'"{NAME}"', f'"{NAME}_transfer"')
                                for f in transfer_flags))
print(TRANSFER_CMD)

## 14. Reading the Training Output

`--error_table="EnergyNacsDipoleMAE"` writes a per-property MAE table at each evaluation
step. The helper below pulls the loss curves out of the log.

In [ ]:
import re, glob

def parse_loss_log(log_path):
    """Extract epoch / train loss / valid loss from an X-MACE log."""
    epochs, train, valid = [], [], []
    pattern = re.compile(
        r'epoch\s+(\d+).*?loss\s+([0-9.eE+\-]+).*?valid_loss\s+([0-9.eE+\-]+)',
        re.IGNORECASE)
    with open(log_path) as fh:
        for line in fh:
            m = pattern.search(line)
            if m:
                epochs.append(int(m.group(1)))
                train.append(float(m.group(2)))
                valid.append(float(m.group(3)))
    return epochs, train, valid


logs = sorted(glob.glob("X-MACE/results/*.log")) + sorted(glob.glob("X-MACE/logs/*.log"))
print("Log files found:", logs or "none — run section 12 first")

if logs:
    epochs, train, valid = parse_loss_log(logs[-1])
    if epochs:
        plt.figure(figsize=(7, 4))
        plt.semilogy(epochs, train, label="Train loss")
        plt.semilogy(epochs, valid, label="Validation loss")
        plt.xlabel("Epoch"); plt.ylabel("Loss (log scale)")
        plt.title("X-MACE training curves"); plt.legend()
        plt.tight_layout(); plt.show()
    else:
        print("Log found but no epoch lines yet — train for more epochs.")
        print("\nLast 20 lines:")
        print("".join(open(logs[-1]).readlines()[-20:]))

## 15. Where to Go Next

* **More epochs.** Five is a demo; real runs are hundreds. Expect under a day on a GPU.
* **Compare the variants.** Train `AutoencoderExcitedMACE` on energies and forces and
  compare the gap region against `ExcitedMACE` — the autoencoder's advantage shows up
  where the surfaces touch.
* **Transfer learning.** Section 13, especially if your excited-state dataset is small.
* **Dynamics.** `SHARC_MACE.py` in `tutorials/` couples a trained model to SHARC for
  surface-hopping trajectories.

**Outputs:** `results/` (logs, error tables) · `checkpoints/` (best model saved automatically)

**References:** [X-MACE paper](https://arxiv.org/abs/2502.12870) ·
[X-MACE GitHub](https://github.com/rhyan10/X-MACE) ·
[MACE docs](https://mace-docs.readthedocs.io)